# Data Quality: HVFHV - 8 Dimensiones - NYC TLC

Evaluacion de las 8 dimensiones de calidad de datos para el conjunto de datos HVFHV (High-Volume For-Hire Vehicle) del NYC Taxi and Limousine Commission.

| Dimension | Descripcion |
|---|---|
| Completitud | Presencia de valores en campos obligatorios |
| Exactitud | Valores concordantes con catalogos conocidos |
| Consistencia | Coherencia logica entre campos relacionados |
| Integridad | Integridad referencial con el catalogo de zonas TLC |
| Razonabilidad | Valores dentro de rangos esperados |
| Oportunidad | Datos dentro del rango temporal valido |
| Unicidad | Ausencia de registros duplicados |
| Validez | Formatos correctos en campos clave |

In [1]:
import os
os.makedirs('images', exist_ok=True)
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, '/home/jovyan/work')

from pyspark.sql import functions as F

# Tipo de vehiculo
VEHICLE = 'hvfhv'
RAW_PATH = f'/home/jovyan/work/data/raw/{VEHICLE}/'

# Diccionario para almacenar resultados de calidad
resultados_dq = {}

print(f'Tipo de vehiculo: {VEHICLE}')
print(f'Ruta de datos: {RAW_PATH}')


Tipo de vehiculo: hvfhv
Ruta de datos: /home/jovyan/work/data/raw/hvfhv/


In [2]:
from src.spark_utils import get_spark, read_parquet

spark = get_spark(app_name=f'DQ_{VEHICLE.upper()}')

# Lectura del conjunto de datos (lee todos los archivos fhvhv_*.parquet de la carpeta)
df = read_parquet(spark, RAW_PATH)

total_registros = df.count()
print(f'Total de registros cargados: {total_registros:,}')
print(f'Columnas disponibles: {df.columns}')
df.printSchema()


2026-07-18 20:22:51 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
2026-07-18 20:22:55 | INFO     | tlc.spark_utils | [SPARK] Read Parquet ← /home/jovyan/work/data/raw/hvfhv/ (Robust Mode)
Total de registros cargados: 715,550,152
Columnas disponibles: ['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_miles', 'trip_time', 'base_passenger_fare', 'tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag', 'cbd_congestion_fee']
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable =

## Dimension 1: Completitud

**Pregunta:** Tenemos todos los datos necesarios?

Se verifica la presencia de valores no nulos en los campos obligatorios del esquema HVFHV.
El score es el promedio del porcentaje de completitud por campo obligatorio.

In [3]:
# Campos obligatorios para HVFHV
campos_obligatorios = [
    'hvfhs_license_num',
    'dispatching_base_num',
    'pickup_datetime',
    'dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'base_passenger_fare',
    'driver_pay'
]

print('--- Dimension 1: Completitud ---')
print(f'Total de registros: {total_registros:,}')
print()

porcentajes_completitud = []
registros_fallidos_completitud = 0

for campo in campos_obligatorios:
    nulos = df.filter(F.col(campo).isNull()).count()
    pct_completo = ((total_registros - nulos) / total_registros) * 100
    porcentajes_completitud.append(pct_completo)
    registros_fallidos_completitud += nulos
    print(f'  Campo "{campo}": {nulos:,} nulos | {pct_completo:.2f}% completo')

# Score: promedio de completitud por campo
score_completitud = sum(porcentajes_completitud) / len(porcentajes_completitud)

resultados_dq['Completitud'] = {
    'score': round(score_completitud, 2),
    'descripcion': 'Porcentaje promedio de completitud en campos obligatorios',
    'registros_fallidos': registros_fallidos_completitud
}

print()
print(f'Score Completitud: {score_completitud:.2f}%')
print(f'Registros fallidos (suma de nulos por campo): {registros_fallidos_completitud:,}')

--- Dimension 1: Completitud ---
Total de registros: 715,550,152

  Campo "hvfhs_license_num": 0 nulos | 100.00% completo
  Campo "dispatching_base_num": 0 nulos | 100.00% completo
  Campo "pickup_datetime": 0 nulos | 100.00% completo
  Campo "dropoff_datetime": 0 nulos | 100.00% completo
  Campo "PULocationID": 0 nulos | 100.00% completo
  Campo "DOLocationID": 0 nulos | 100.00% completo
  Campo "base_passenger_fare": 0 nulos | 100.00% completo
  Campo "driver_pay": 0 nulos | 100.00% completo

Score Completitud: 100.00%
Registros fallidos (suma de nulos por campo): 0


## Dimension 2: Exactitud

**Pregunta:** Los valores son correctos?

Se verifica que los valores coincidan con los catalogos conocidos del TLC:
- `hvfhs_license_num`: debe estar en ['HV0002', 'HV0003', 'HV0004', 'HV0005']
- `shared_request_flag`: debe ser 'Y' o 'N'
- `shared_match_flag`: debe ser 'Y' o 'N'
- `access_a_ride_flag`: debe ser 'Y' o 'N'
- `wav_request_flag`: debe ser 'Y' o 'N'
- `wav_match_flag`: debe ser 'Y' o 'N'

In [4]:
print('--- Dimension 2: Exactitud ---')
print()

LICENCIAS_VALIDAS = ['HV0002', 'HV0003', 'HV0004', 'HV0005']
FLAGS_VALIDOS = ['Y', 'N']
COLUMNAS_FLAG = [
    'shared_request_flag',
    'shared_match_flag',
    'access_a_ride_flag',
    'wav_request_flag',
    'wav_match_flag'
]

# Construir condicion conjunta de fallas de banderas
condicion_falla_flags = F.lit(False)
for col_flag in COLUMNAS_FLAG:
    condicion_falla_flags = condicion_falla_flags | (
        F.col(col_flag).isNotNull() & (~F.col(col_flag).isin(FLAGS_VALIDOS))
    )

# Condicion conjunta de exactitud
condicion_falla_exactitud = (
    F.col('hvfhs_license_num').isNotNull() &
    (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS))
) | condicion_falla_flags

# --- OPTIMIZACION ONE-PASS ---
_exprs = [
    F.sum(F.when(F.col('hvfhs_license_num').isNotNull() & (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS)), 1).otherwise(0)).alias('falla_licencia'),
    F.sum(F.when(condicion_falla_exactitud, 1).otherwise(0)).alias('registros_fallidos_exactitud')
]
# Agregamos las expresiones para los flags
for col_flag in COLUMNAS_FLAG:
    _exprs.append(
        F.sum(F.when(F.col(col_flag).isNotNull() & (~F.col(col_flag).isin(FLAGS_VALIDOS)), 1).otherwise(0)).alias(f'falla_flag_{col_flag}')
    )

_res = df.agg(*_exprs).collect()[0]
falla_licencia = _res['falla_licencia'] or 0
registros_fallidos_exactitud = _res['registros_fallidos_exactitud'] or 0

print(f'  hvfhs_license_num fuera de catalogo {LICENCIAS_VALIDAS}: {falla_licencia:,} registros')

for col_flag in COLUMNAS_FLAG:
    falla_flag = _res[f'falla_flag_{col_flag}'] or 0
    print(f'  {col_flag} con valor invalido (no Y ni N): {falla_flag:,} registros')

score_exactitud = ((total_registros - registros_fallidos_exactitud) / total_registros) * 100

resultados_dq['Exactitud'] = {
    'score': round(score_exactitud, 2),
    'descripcion': 'Porcentaje de registros cuyos valores coinciden con catalogos TLC (licencia y banderas)',
    'registros_fallidos': registros_fallidos_exactitud
}

print()
print(f'Score Exactitud: {score_exactitud:.2f}%')
print(f'Registros fallidos: {registros_fallidos_exactitud:,}')


--- Dimension 2: Exactitud ---

  hvfhs_license_num fuera de catalogo ['HV0002', 'HV0003', 'HV0004', 'HV0005']: 0 registros
  shared_request_flag con valor invalido (no Y ni N): 0 registros
  shared_match_flag con valor invalido (no Y ni N): 0 registros
  access_a_ride_flag con valor invalido (no Y ni N): 134,931,167 registros
  wav_request_flag con valor invalido (no Y ni N): 0 registros
  wav_match_flag con valor invalido (no Y ni N): 0 registros

Score Exactitud: 81.14%
Registros fallidos: 134,931,167


## Dimension 3: Consistencia

**Pregunta:** Los datos son coherentes entre si?

Se verifica la coherencia logica entre campos relacionados en HVFHV:
- `request_datetime` <= `pickup_datetime` <= `dropoff_datetime`
- `driver_pay` <= `base_passenger_fare` * 1.5 (participacion razonable del conductor)
- `trip_time` > 0 cuando `trip_miles` > 0

In [5]:
print('--- Dimension 3: Consistencia ---')
print()

# Regla 1: request_datetime <= pickup_datetime

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('request_datetime').isNotNull() & F.col('pickup_datetime').isNotNull() & (F.col('request_datetime') > F.col('pickup_datetime')), 1).otherwise(0)).alias('falla_request_pickup'),
    F.sum(F.when(F.col('pickup_datetime').isNotNull() & F.col('dropoff_datetime').isNotNull() & (F.col('pickup_datetime') >= F.col('dropoff_datetime')), 1).otherwise(0)).alias('falla_orden_fechas'),
    F.sum(F.when(F.col('driver_pay').isNotNull() & F.col('base_passenger_fare').isNotNull() & (F.col('base_passenger_fare') > 0) & (F.col('driver_pay') > F.col('base_passenger_fare') * 1.5), 1).otherwise(0)).alias('falla_driver_pay'),
    F.sum(F.when(F.col('trip_miles').isNotNull() & F.col('trip_time').isNotNull() & (F.col('trip_miles') > 0) & (F.col('trip_time') <= 0), 1).otherwise(0)).alias('falla_trip_time_miles')
]
_res = df.agg(*_exprs).collect()[0]
falla_request_pickup = _res['falla_request_pickup'] or 0
falla_orden_fechas = _res['falla_orden_fechas'] or 0
falla_driver_pay = _res['falla_driver_pay'] or 0
falla_trip_time_miles = _res['falla_trip_time_miles'] or 0
# ---------------------------------------------
# falla_request_pickup (Optimizado en One-Pass)
print(f'  request_datetime > pickup_datetime: {falla_request_pickup:,} registros')

# Regla 2: pickup_datetime <= dropoff_datetime
# falla_orden_fechas (Optimizado en One-Pass)
print(f'  pickup_datetime >= dropoff_datetime (orden incorrecto): {falla_orden_fechas:,} registros')

# Regla 3: driver_pay <= base_passenger_fare * 1.5
# falla_driver_pay (Optimizado en One-Pass)
print(f'  driver_pay > base_passenger_fare * 1.5: {falla_driver_pay:,} registros')

# Regla 4: trip_time > 0 cuando trip_miles > 0
# falla_trip_time_miles (Optimizado en One-Pass)
print(f'  trip_time <= 0 cuando trip_miles > 0: {falla_trip_time_miles:,} registros')

# Registros que fallan CUALQUIERA de las reglas de consistencia
df_falla_consistencia = df.filter(
    (
        F.col('request_datetime').isNotNull() &
        F.col('pickup_datetime').isNotNull() &
        (F.col('request_datetime') > F.col('pickup_datetime'))
    ) |
    (
        F.col('pickup_datetime').isNotNull() &
        F.col('dropoff_datetime').isNotNull() &
        (F.col('pickup_datetime') >= F.col('dropoff_datetime'))
    ) |
    (
        F.col('driver_pay').isNotNull() &
        F.col('base_passenger_fare').isNotNull() &
        (F.col('base_passenger_fare') > 0) &
        (F.col('driver_pay') > F.col('base_passenger_fare') * 1.5)
    ) |
    (
        F.col('trip_miles').isNotNull() &
        F.col('trip_time').isNotNull() &
        (F.col('trip_miles') > 0) &
        (F.col('trip_time') <= 0)
    )
)
registros_fallidos_consistencia = df_falla_consistencia.count()
score_consistencia = ((total_registros - registros_fallidos_consistencia) / total_registros) * 100

resultados_dq['Consistencia'] = {
    'score': round(score_consistencia, 2),
    'descripcion': 'Porcentaje de registros con coherencia logica entre fechas, tarifas y kilometraje',
    'registros_fallidos': registros_fallidos_consistencia
}

print()
print(f'Score Consistencia: {score_consistencia:.2f}%')
print(f'Registros fallidos: {registros_fallidos_consistencia:,}')

--- Dimension 3: Consistencia ---

  request_datetime > pickup_datetime: 7,390,312 registros
  pickup_datetime >= dropoff_datetime (orden incorrecto): 30,418 registros
  driver_pay > base_passenger_fare * 1.5: 6,352,127 registros
  trip_time <= 0 cuando trip_miles > 0: 4 registros

Score Consistencia: 98.12%
Registros fallidos: 13,487,763


## Dimension 4: Integridad

**Pregunta:** Se mantienen las relaciones entre tablas?

Se verifica la integridad referencial con el catalogo de zonas TLC y el catalogo de licencias:
- `PULocationID` debe estar en el rango 1-265 (zonas TLC validas) - UPPERCASE en HVFHV
- `DOLocationID` debe estar en el rango 1-265 (zonas TLC validas) - UPPERCASE en HVFHV
- `hvfhs_license_num` debe estar en el catalogo de licencias HVFHV validas

In [6]:
print('--- Dimension 4: Integridad ---')
print()

# Rango valido de zonas TLC: 1 a 265
ZONA_MIN = 1
ZONA_MAX = 265
LICENCIAS_VALIDAS = ['HV0002', 'HV0003', 'HV0004', 'HV0005']

# Regla 1: PULocationID debe estar en rango valido (nota: UPPERCASE en HVFHV)

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('PULocationID').isNotNull() & ( (F.col('PULocationID') < ZONA_MIN) | (F.col('PULocationID') > ZONA_MAX) ), 1).otherwise(0)).alias('falla_pu'),
    F.sum(F.when(F.col('DOLocationID').isNotNull() & ( (F.col('DOLocationID') < ZONA_MIN) | (F.col('DOLocationID') > ZONA_MAX) ), 1).otherwise(0)).alias('falla_do'),
    F.sum(F.when(F.col('hvfhs_license_num').isNotNull() & (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS)), 1).otherwise(0)).alias('falla_licencia_integridad')
]
_res = df.agg(*_exprs).collect()[0]
falla_pu = _res['falla_pu'] or 0
falla_do = _res['falla_do'] or 0
falla_licencia_integridad = _res['falla_licencia_integridad'] or 0
# ---------------------------------------------
# falla_pu (Optimizado en One-Pass)
print(f'  PULocationID fuera de rango [1-265]: {falla_pu:,} registros')

# Regla 2: DOLocationID debe estar en rango valido (nota: UPPERCASE en HVFHV)
# falla_do (Optimizado en One-Pass)
print(f'  DOLocationID fuera de rango [1-265]: {falla_do:,} registros')

# Regla 3: hvfhs_license_num en catalogo de licencias validas
# falla_licencia_integridad (Optimizado en One-Pass)
print(f'  hvfhs_license_num fuera del catalogo HVFHV: {falla_licencia_integridad:,} registros')

# Registros que fallan CUALQUIERA de las reglas de integridad
df_falla_integridad = df.filter(
    (
        F.col('PULocationID').isNotNull() &
        ((F.col('PULocationID') < ZONA_MIN) | (F.col('PULocationID') > ZONA_MAX))
    ) |
    (
        F.col('DOLocationID').isNotNull() &
        ((F.col('DOLocationID') < ZONA_MIN) | (F.col('DOLocationID') > ZONA_MAX))
    ) |
    (
        F.col('hvfhs_license_num').isNotNull() &
        (~F.col('hvfhs_license_num').isin(LICENCIAS_VALIDAS))
    )
)
registros_fallidos_integridad = df_falla_integridad.count()
score_integridad = ((total_registros - registros_fallidos_integridad) / total_registros) * 100

resultados_dq['Integridad'] = {
    'score': round(score_integridad, 2),
    'descripcion': 'Porcentaje de registros con IDs de zona en rango TLC valido y licencia en catalogo',
    'registros_fallidos': registros_fallidos_integridad
}

print()
print(f'Score Integridad: {score_integridad:.2f}%')
print(f'Registros fallidos: {registros_fallidos_integridad:,}')

--- Dimension 4: Integridad ---

  PULocationID fuera de rango [1-265]: 0 registros
  DOLocationID fuera de rango [1-265]: 0 registros
  hvfhs_license_num fuera del catalogo HVFHV: 0 registros

Score Integridad: 100.00%
Registros fallidos: 0


## Dimension 5: Razonabilidad

**Pregunta:** Los valores estan dentro de rangos esperados?

Se verifica que los valores numericos clave esten dentro de rangos razonables para HVFHV:
- `trip_miles`: entre 0 y 300 millas (Uber/Lyft pueden tener viajes largos)
- `base_passenger_fare`: entre 0 y 1000 dolares
- `driver_pay`: entre 0 y 800 dolares
- `trip_time`: entre 60 y 43,200 segundos (1 minuto a 12 horas)
- `tips`: mayor o igual a 0

In [7]:
print('--- Dimension 5: Razonabilidad ---')
print()

# Regla 1: trip_miles entre 0 y 300

# --- OPTIMIZACION ONE-PASS (Auto-generada) ---
_exprs = [
    F.sum(F.when(F.col('trip_miles').isNotNull() & ( (F.col('trip_miles') < 0) | (F.col('trip_miles') > 300) ), 1).otherwise(0)).alias('falla_millas'),
    F.sum(F.when(F.col('base_passenger_fare').isNotNull() & ( (F.col('base_passenger_fare') < 0) | (F.col('base_passenger_fare') > 1000) ), 1).otherwise(0)).alias('falla_tarifa'),
    F.sum(F.when(F.col('driver_pay').isNotNull() & ( (F.col('driver_pay') < 0) | (F.col('driver_pay') > 800) ), 1).otherwise(0)).alias('falla_driver_pay'),
    F.sum(F.when(F.col('trip_time').isNotNull() & ( (F.col('trip_time') < 60) | (F.col('trip_time') > 43200) ), 1).otherwise(0)).alias('falla_trip_time'),
    F.sum(F.when(F.col('tips').isNotNull() & (F.col('tips') < 0), 1).otherwise(0)).alias('falla_tips')
]
_res = df.agg(*_exprs).collect()[0]
falla_millas = _res['falla_millas'] or 0
falla_tarifa = _res['falla_tarifa'] or 0
falla_driver_pay = _res['falla_driver_pay'] or 0
falla_trip_time = _res['falla_trip_time'] or 0
falla_tips = _res['falla_tips'] or 0
# ---------------------------------------------
# falla_millas (Optimizado en One-Pass)
print(f'  trip_miles fuera de rango [0-300]: {falla_millas:,} registros')

# Regla 2: base_passenger_fare entre 0 y 1000
# falla_tarifa (Optimizado en One-Pass)
print(f'  base_passenger_fare fuera de rango [0-1000]: {falla_tarifa:,} registros')

# Regla 3: driver_pay entre 0 y 800
# falla_driver_pay (Optimizado en One-Pass)
print(f'  driver_pay fuera de rango [0-800]: {falla_driver_pay:,} registros')

# Regla 4: trip_time entre 60 y 43200 segundos
# falla_trip_time (Optimizado en One-Pass)
print(f'  trip_time fuera de rango [60-43200 segundos]: {falla_trip_time:,} registros')

# Regla 5: tips >= 0
# falla_tips (Optimizado en One-Pass)
print(f'  tips negativos: {falla_tips:,} registros')

# Registros que fallan CUALQUIERA de las reglas de razonabilidad
df_falla_razon = df.filter(
    (
        F.col('trip_miles').isNotNull() &
        ((F.col('trip_miles') < 0) | (F.col('trip_miles') > 300))
    ) |
    (
        F.col('base_passenger_fare').isNotNull() &
        ((F.col('base_passenger_fare') < 0) | (F.col('base_passenger_fare') > 1000))
    ) |
    (
        F.col('driver_pay').isNotNull() &
        ((F.col('driver_pay') < 0) | (F.col('driver_pay') > 800))
    ) |
    (
        F.col('trip_time').isNotNull() &
        ((F.col('trip_time') < 60) | (F.col('trip_time') > 43200))
    ) |
    (
        F.col('tips').isNotNull() &
        (F.col('tips') < 0)
    )
)
registros_fallidos_razon = df_falla_razon.count()
score_razonabilidad = ((total_registros - registros_fallidos_razon) / total_registros) * 100

resultados_dq['Razonabilidad'] = {
    'score': round(score_razonabilidad, 2),
    'descripcion': 'Porcentaje de registros con valores numericos dentro de rangos razonables',
    'registros_fallidos': registros_fallidos_razon
}

print()
print(f'Score Razonabilidad: {score_razonabilidad:.2f}%')
print(f'Registros fallidos: {registros_fallidos_razon:,}')

--- Dimension 5: Razonabilidad ---

  trip_miles fuera de rango [0-300]: 152 registros
  base_passenger_fare fuera de rango [0-1000]: 100,883 registros
  driver_pay fuera de rango [0-800]: 6,091 registros
  trip_time fuera de rango [60-43200 segundos]: 58,714 registros
  tips negativos: 0 registros

Score Razonabilidad: 99.98%
Registros fallidos: 163,580


## Dimension 6: Oportunidad

**Pregunta:** Los datos estan actualizados?

Se verifica que los registros correspondan al rango temporal esperado para el conjunto de datos HVFHV:
- Rango valido: 2019-2025 (Ley HVFHV entro en vigor en febrero de 2019)
- Se contabilizan registros por anio para identificar la distribucion temporal

In [8]:
print('--- Dimension 6: Oportunidad ---')
print()

ANIO_MIN = 2019
ANIO_MAX = 2025

# Extraer anio del pickup_datetime
df_con_anio = df.withColumn('anio_recogida', F.year('pickup_datetime'))

# Distribucion por anio
print('  Distribucion de registros por anio de recogida:')
dist_anio = df_con_anio.groupBy('anio_recogida').count().orderBy('anio_recogida')
dist_anio.show(truncate=False)

# Registros fuera del rango valido
registros_fallidos_oportunidad = df_con_anio.filter(
    F.col('anio_recogida').isNull() |
    (F.col('anio_recogida') < ANIO_MIN) |
    (F.col('anio_recogida') > ANIO_MAX)
).count()

score_oportunidad = ((total_registros - registros_fallidos_oportunidad) / total_registros) * 100

resultados_dq['Oportunidad'] = {
    'score': round(score_oportunidad, 2),
    'descripcion': f'Porcentaje de registros con anio de recogida en el rango [{ANIO_MIN}-{ANIO_MAX}]',
    'registros_fallidos': registros_fallidos_oportunidad
}

print(f'Registros fuera del rango [{ANIO_MIN}-{ANIO_MAX}]: {registros_fallidos_oportunidad:,}')
print()
print(f'Score Oportunidad: {score_oportunidad:.2f}%')

--- Dimension 6: Oportunidad ---

  Distribucion de registros por anio de recogida:
+-------------+---------+
|anio_recogida|count    |
+-------------+---------+
|2023         |232490020|
|2024         |239470448|
|2025         |243589684|
+-------------+---------+

Registros fuera del rango [2019-2025]: 0

Score Oportunidad: 100.00%


## Dimension 7: Unicidad

**Pregunta:** Existen registros duplicados?

Se detectan registros duplicados agrupando por la combinacion de campos que identifican unicamente un viaje HVFHV:
- `pickup_datetime`
- `dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `hvfhs_license_num`
- `base_passenger_fare`

In [9]:
print('--- Dimension 7: Unicidad ---')
print()

# Campos que definen la unicidad de un viaje HVFHV
campos_unicidad = [
    'pickup_datetime',
    'dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'hvfhs_license_num',
    'base_passenger_fare'
]

# Contar ocurrencias por combinacion de campos
df_agrupado = df.groupBy(campos_unicidad).count()

# Registros duplicados: grupos con mas de 1 ocurrencia
df_duplicados = df_agrupado.filter(F.col('count') > 1)
grupos_duplicados = df_duplicados.count()

# Total de registros excedentes que son duplicados
excedentes_resultado = df_duplicados.agg(
    F.sum(F.col('count') - 1).alias('excedentes')
).collect()[0]['excedentes']

registros_duplicados_total = int(excedentes_resultado) if excedentes_resultado is not None else 0

registros_unicos = total_registros - registros_duplicados_total
score_unicidad = (registros_unicos / total_registros) * 100

print(f'  Grupos de registros duplicados encontrados: {grupos_duplicados:,}')
print(f'  Total de registros excedentes (duplicados): {registros_duplicados_total:,}')
print(f'  Registros unicos: {registros_unicos:,}')

resultados_dq['Unicidad'] = {
    'score': round(score_unicidad, 2),
    'descripcion': 'Porcentaje de registros unicos (sin duplicados) por combinacion de campos clave',
    'registros_fallidos': registros_duplicados_total
}

print()
print(f'Score Unicidad: {score_unicidad:.2f}%')

--- Dimension 7: Unicidad ---

  Grupos de registros duplicados encontrados: 4,577
  Total de registros excedentes (duplicados): 4,577
  Registros unicos: 715,545,575

Score Unicidad: 100.00%


## Dimension 8: Validez

**Pregunta:** Los formatos son correctos?

Se verifica que los campos clave tengan formatos validos en HVFHV:
- `dispatching_base_num`: debe comenzar con 'B' (validacion de formato)
- `hvfhs_license_num`: debe coincidir con el patron HV seguido de 4 digitos (HV[0-9]{4})
- Columnas de banderas (`shared_request_flag`, etc.): deben tener exactamente 1 caracter
- `trip_time`: debe ser un entero (sin decimales que indiquen problemas de tipo)

In [11]:
print('--- Dimension 8: Validez ---')
print()

COLUMNAS_FLAG = [
    'shared_request_flag',
    'shared_match_flag',
    'access_a_ride_flag',
    'wav_request_flag',
    'wav_match_flag'
]

# Construir condicion de validez conjunta para las banderas
condicion_falla_long_flags = F.lit(False)
for col_flag in COLUMNAS_FLAG:
    condicion_falla_long_flags = condicion_falla_long_flags | (
        F.col(col_flag).isNotNull() & (F.length(F.col(col_flag)) != 1)
    )

# Registros que fallan CUALQUIERA de las reglas de validez
condicion_falla_validez = (
    (
        F.col('dispatching_base_num').isNotNull() &
        (~F.col('dispatching_base_num').startswith('B'))
    ) |
    (
        F.col('hvfhs_license_num').isNotNull() &
        (~F.col('hvfhs_license_num').rlike('^HV[0-9]{4}$'))
    ) |
    condicion_falla_long_flags |
    (
        F.col('trip_time').isNotNull() &
        (F.col('trip_time') != F.floor(F.col('trip_time')))
    )
)

# --- OPTIMIZACION ONE-PASS ---
_exprs = [
    F.sum(F.when(F.col('dispatching_base_num').isNotNull() & (~F.col('dispatching_base_num').startswith('B')), 1).otherwise(0)).alias('falla_formato_base'),
    F.sum(F.when(F.col('hvfhs_license_num').isNotNull() & (~F.col('hvfhs_license_num').rlike('^HV[0-9]{4}$')), 1).otherwise(0)).alias('falla_patron_licencia'),
    F.sum(F.when(F.col('trip_time').isNotNull() & (F.col('trip_time') != F.floor(F.col('trip_time'))), 1).otherwise(0)).alias('falla_trip_time_decimal'),
    F.sum(F.when(condicion_falla_validez, 1).otherwise(0)).alias('registros_fallidos_validez')
]

# Agregar una expresion dinamica por cada bandera para saber en cual fallo
for col_flag in COLUMNAS_FLAG:
    _exprs.append(
        F.sum(F.when(F.col(col_flag).isNotNull() & (F.length(F.col(col_flag)) != 1), 1).otherwise(0)).alias(f'falla_len_{col_flag}')
    )

# Calcular todas las evaluaciones (incluyendo los flags) en UNA SOLA PASADA
_res = df.agg(*_exprs).collect()[0]

falla_formato_base = _res['falla_formato_base'] or 0
falla_patron_licencia = _res['falla_patron_licencia'] or 0
falla_trip_time_decimal = _res['falla_trip_time_decimal'] or 0
registros_fallidos_validez = _res['registros_fallidos_validez'] or 0

# --- Impresion de resultados ---
print(f'  dispatching_base_num no comienza con "B": {falla_formato_base:,} registros')
print(f'  hvfhs_license_num no coincide con patron HV[0-9]{{4}}: {falla_patron_licencia:,} registros')

for col_flag in COLUMNAS_FLAG:
    falla_len_flag = _res[f'falla_len_{col_flag}'] or 0
    print(f'  {col_flag} con longitud != 1: {falla_len_flag:,} registros')

print(f'  trip_time con parte decimal (no entero): {falla_trip_time_decimal:,} registros')

score_validez = ((total_registros - registros_fallidos_validez) / total_registros) * 100

resultados_dq['Validez'] = {
    'score': round(score_validez, 2),
    'descripcion': 'Porcentaje de registros con formatos correctos en campos clave (base, licencia, banderas)',
    'registros_fallidos': registros_fallidos_validez
}

print()
print(f'Score Validez: {score_validez:.2f}%')
print(f'Registros fallidos: {registros_fallidos_validez:,}')

--- Dimension 8: Validez ---

  dispatching_base_num no comienza con "B": 0 registros
  hvfhs_license_num no coincide con patron HV[0-9]{4}: 0 registros
  shared_request_flag con longitud != 1: 0 registros
  shared_match_flag con longitud != 1: 0 registros
  access_a_ride_flag con longitud != 1: 0 registros
  wav_request_flag con longitud != 1: 0 registros
  wav_match_flag con longitud != 1: 0 registros
  trip_time con parte decimal (no entero): 0 registros

Score Validez: 100.00%
Registros fallidos: 0


## Resumen Final: Puntuacion Global de Calidad de Datos

Se presenta el resumen de las 8 dimensiones evaluadas con su puntuacion individual y la puntuacion global promedio.

**Escala de colores:**
- Verde: score >= 95% (calidad alta)
- Naranja: score >= 80% (calidad media)
- Rojo: score < 80% (calidad baja)

In [12]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType

print('=' * 60)
print('RESUMEN FINAL: CALIDAD DE DATOS HVFHV')
print('=' * 60)
print()

# Construir lista de resultados
filas_resumen = []
for dimension, valores in resultados_dq.items():
    filas_resumen.append((dimension, float(valores['score']), int(valores['registros_fallidos'])))
    print(f'  {dimension:<18} | Score: {valores["score"]:6.2f}% | Fallidos: {valores["registros_fallidos"]:>12,}')

print()

# Score global (promedio de las 8 dimensiones)
scores = [v['score'] for v in resultados_dq.values()]
score_global = sum(scores) / len(scores)
print(f'  Score Global (promedio): {score_global:.2f}%')
print()

# Crear DataFrame de resumen con PySpark
schema_resumen = StructType([
    StructField('Dimension', StringType(), True),
    StructField('Score', FloatType(), True),
    StructField('Registros_Fallidos', IntegerType(), True)
])
df_resumen = spark.createDataFrame(filas_resumen, schema=schema_resumen)
print('Tabla de resultados:')
df_resumen.show(truncate=False)

# --- Grafico de barras horizontales ---
dimensiones = [f[0] for f in filas_resumen]
scores_lista = [f[1] for f in filas_resumen]

colores = []
for s in scores_lista:
    if s >= 95:
        colores.append('#2ecc71')   # Verde
    elif s >= 80:
        colores.append('#f39c12')   # Naranja
    else:
        colores.append('#e74c3c')   # Rojo

fig, ax = plt.subplots(figsize=(11, 6))
y_pos = range(len(dimensiones))
barras = ax.barh(y_pos, scores_lista, color=colores, edgecolor='white', height=0.6)

# Etiquetas de valor en las barras
for barra, score in zip(barras, scores_lista):
    ax.text(
        barra.get_width() + 0.3,
        barra.get_y() + barra.get_height() / 2,
        f'{score:.1f}%',
        va='center',
        ha='left',
        fontsize=10,
        fontweight='bold'
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(dimensiones, fontsize=11)
ax.set_xlabel('Score de Calidad (%)', fontsize=11)
ax.set_title(
    f'Calidad de Datos HVFHV - 8 Dimensiones\nScore Global: {score_global:.2f}%',
    fontsize=13,
    fontweight='bold'
)
ax.set_xlim(0, 108)
ax.axvline(x=95, color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1)
ax.axvline(x=80, color='#f39c12', linestyle='--', alpha=0.6, linewidth=1)

# Leyenda
leyenda = [
    mpatches.Patch(color='#2ecc71', label='Alta (>= 95%)'),
    mpatches.Patch(color='#f39c12', label='Media (>= 80%)'),
    mpatches.Patch(color='#e74c3c', label='Baja (< 80%)')
]
ax.legend(handles=leyenda, loc='lower right', fontsize=9)

ax.invert_yaxis()
plt.tight_layout()
plt.savefig('/home/jovyan/work/notebooks/quality/hvfhv/images/dq_score_hvfhv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico guardado en: /home/jovyan/work/notebooks/quality/hvfhv/images/dq_score_hvfhv.png')

spark.stop()
print()
print('Sesion de Spark finalizada.')


RESUMEN FINAL: CALIDAD DE DATOS HVFHV

  Completitud        | Score: 100.00% | Fallidos:            0
  Exactitud          | Score:  81.14% | Fallidos:  134,931,167
  Consistencia       | Score:  98.12% | Fallidos:   13,487,763
  Integridad         | Score: 100.00% | Fallidos:            0
  Razonabilidad      | Score:  99.98% | Fallidos:      163,580
  Oportunidad        | Score: 100.00% | Fallidos:            0
  Unicidad           | Score: 100.00% | Fallidos:        4,577
  Validez            | Score: 100.00% | Fallidos:            0

  Score Global (promedio): 97.41%

Tabla de resultados:
+-------------+-----+------------------+
|Dimension    |Score|Registros_Fallidos|
+-------------+-----+------------------+
|Completitud  |100.0|0                 |
|Exactitud    |81.14|134931167         |
|Consistencia |98.12|13487763          |
|Integridad   |100.0|0                 |
|Razonabilidad|99.98|163580            |
|Oportunidad  |100.0|0                 |
|Unicidad     |100.0|4577      